# Club & Country — the league dependency map

Where does each 2026 World Cup squad actually earn its living? This notebook
maps every one of the 1,246 players to the club — and league/country — they
play for, then turns that into flows (nation → league) and a **domestic
dependency** stat per team.

The modelling is light; the **data engineering** is the point. The same club
appears as "Manchester City F.C.", "FC Bayern Munich", "Al Hilal SFC" — messy,
inconsistent entities that have to be reconciled before a single number is
trustworthy. Logic lives in `mlfootball/squads.py`; source is the Wikipedia
2026 squad tables (raw wikitext, parsed — no scraping).

In [1]:
from pathlib import Path
import json
from mlfootball import squads as S

players = S.load_players()
print(f"{len(players)} players · {len(set(p['nation'] for p in players))} nations · "
      f"{len(set(p['code'] for p in players))} club-countries")

1246 players · 48 nations · 71 club-countries


## 1 · Entity resolution — cleaning the clubs

Raw club strings carry club-form noise (F.C./FC/AFC/SC/CF/AS/SSC…). We peel it
off to land on clean, consistent labels — the difference between one node and
three for the same club.

In [2]:
for ex in ["Manchester City F.C.", "FC Bayern Munich", "Paris Saint-Germain FC",
           "Al Hilal SFC", "AS Monaco", "SSC Napoli", "TSG 1899 Hoffenheim"]:
    print(f"  {ex:24} → {S.normalize_club(ex)}")
print(f"\n{len(set(p['club_raw'] for p in players))} raw club strings → "
      f"{len(set(p['club'] for p in players))} normalised clubs")

  Manchester City F.C.     → Manchester City
  FC Bayern Munich         → Bayern Munich
  Paris Saint-Germain FC   → Paris Saint-Germain
  Al Hilal SFC             → Al Hilal
  AS Monaco                → Monaco
  SSC Napoli               → Napoli
  TSG 1899 Hoffenheim      → 1899 Hoffenheim

451 raw club strings → 450 normalised clubs


## 2 · Domestic dependency — who plays at home?

Share of each squad employed in its own country's league.

In [3]:
nb = S.nation_breakdowns(players)
print("Most home-based:")
for r in nb[:5]:
    print(f"  {r['nation']:14} {r['domestic_pct']:5}%  ({r['domestic']}/{r['n_players']})")
print("\nMost exported (spread across the most leagues):")
for r in sorted(nb, key=lambda x: -x["n_leagues"])[:5]:
    print(f"  {r['nation']:14} {r['domestic_pct']:5}% domestic · {r['n_leagues']} leagues")

Most home-based:
  Qatar           96.2%  (25/26)
  Saudi Arabia    96.2%  (25/26)
  England         80.8%  (21/26)
  South Africa    73.1%  (19/26)
  Germany         73.1%  (19/26)

Most exported (spread across the most leagues):
  Panama           7.7% domestic · 17 leagues
  Bosnia and Herzegovina   3.8% domestic · 17 leagues
  Iraq            38.5% domestic · 15 leagues
  Haiti            3.8% domestic · 15 leagues
  South Korea     26.9% domestic · 14 leagues


Two clean poles: the Gulf hosts (**Qatar, Saudi Arabia ~96%**) keep their squads
home, and **England (~81%)** sits in the Premier League's gravity well — while
diaspora and smaller nations (**Panama, Bosnia, Haiti**) scatter across 15+
leagues with almost nobody playing at home.

## 3 · The flows — nation → league

In [4]:
sk = S.sankey_league(players)
print(f"league Sankey: {len(sk['nodes'])} nodes, {len(sk['links'])} links")
print("destination leagues:", ", ".join(sk["leagues"]))
print("\nbiggest leagues by World Cup players:")
for lg in S.league_leaderboard(players)[:8]:
    print(f"  {lg['league']:22} {lg['count']}")

league Sankey: 64 nodes, 322 links
destination leagues: Premier League, Bundesliga, La Liga, Ligue 1, Serie A, Saudi Pro League, Süper Lig, MLS, Eredivisie, Primeira Liga, Brasileirão, Belgian Pro League, Qatar Stars League, Liga MX, Persian Gulf Pro League, Other leagues

biggest leagues by World Cup players:
  Premier League         200
  Bundesliga             108
  La Liga                86
  Ligue 1                85
  Serie A                71
  Saudi Pro League       49
  Süper Lig              45
  MLS                    43


## 4 · Export for the web app

In [5]:
# Ship the per-nation breakdowns (the source of truth) + the league ranking; the
# web app builds both the league- and club-level Sankeys from these client-side.
out = {
    "n_players": len(players),
    "nations": nb,
    "league_leaderboard": S.league_leaderboard(players),
}
dest = Path(S.__file__).resolve().parent.parent / "site" / "data" / "squads.json"
dest.write_text(json.dumps(out, ensure_ascii=False))
print(f"wrote {dest} ({dest.stat().st_size/1024:.1f} KB)")

wrote /Users/williamcatt/Documents/Projects/worldcup2026/site/data/squads.json (124.6 KB)
